In [ ]:
#==================================
# DEM 提取脚本 - 终极多线程修复版
# 解决：AttributeError 与 Hillshade 问题
#==================================

import arcpy
import os
import time
import sys
from concurrent.futures import ThreadPoolExecutor
import threading

# ==================== 1. 参数设置 ====================
service_url = "https://elevation.nationalmap.gov/arcgis/services/3DEPElevation/ImageServer"
buffer_layer = r"C:\Users\King\Documents\ArcGIS\Projects\MyProject8\MyProject8.gdb\GenerateTessellation_1_ExportFeatures2"
out_folder = r"C:\Users\King\Desktop\大三下学习生活\prj-Alban\data\raw\DEM\DEM_Evelation_Final"

USE_MULTITHREAD = True  
MAX_WORKERS = 6         # 建议 6-8，防止被服务器封锁

# ==================== 2. 环境初始化 ====================
if not os.path.exists(out_folder):
    os.makedirs(out_folder)

arcpy.env.overwriteOutput = True
arcpy.env.cellSize = 1 
arcpy.env.extent = None 

# 确保获取完整点数
arcpy.management.SelectLayerByAttribute(buffer_layer, "CLEAR_SELECTION")
total_points = int(arcpy.management.GetCount(buffer_layer)[0])

success_count = 0
skip_count = 0
error_count = 0
lock = threading.Lock() 
start_time = time.time()

# ==================== 3. 核心下载函数 ====================
def download_dem_task(oid, wkt):
    global success_count, skip_count, error_count
    
    out_path = os.path.join(out_folder, f"Extract_DEM_{oid}.tif")
    if os.path.exists(out_path):
        with lock:
            skip_count += 1
        return

    try:
        # 恢复几何对象
        feat_geom = arcpy.FromWKT(wkt)
        
        # 【关键修复】：在线程内使用 arcpy.sa.Apply 动态包装 URL
        # 强制指定 'None' 模板以获取原始高程数据 [cite: 2026-02-21]
        raw_raster = arcpy.sa.Apply(service_url, "None")
        
        # 执行提取
        extracted = arcpy.sa.ExtractByMask(raw_raster, feat_geom)
        extracted.save(out_path)
        
        with lock:
            success_count += 1
            # 每 5 个点更新一次进度，减少终端刷新压力
            if (success_count + skip_count) % 5 == 0 or (success_count + skip_count) == total_points:
                elapsed = time.time() - start_time
                speed = (success_count + skip_count) / elapsed
                remaining = (total_points - (success_count + skip_count)) / speed / 60
                print(f"进度: {success_count + skip_count}/{total_points} | 速度: {speed:.2f} 点/秒 | 预计剩余: {remaining:.1f} 分钟")

    except Exception as e:
        with lock:
            error_count += 1
            print(f"!!! OID {oid} 失败: {e}")

# ==================== 4. 任务执行逻辑 ====================
print(f"--- 任务启动：共计 {total_points} 个样点 (多线程: {USE_MULTITHREAD}) ---")

tasks = []
# 预先读取所有点位数据
with arcpy.da.SearchCursor(buffer_layer, ["OID@", "SHAPE@WKT"]) as cursor:
    for row in cursor:
        tasks.append((row[0], row[1]))

if USE_MULTITHREAD:
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        # 将任务提交到线程池
        for oid, wkt in tasks:
            executor.submit(download_dem_task, oid, wkt)
else:
    for oid, wkt in tasks:
        download_dem_task(oid, wkt)

# ==================== 5. 最终报告 ====================
total_time = (time.time() - start_time) / 60
print(f"\n--- 任务结束 ---")
print(f"成功: {success_count} | 跳过: {skip_count} | 失败: {error_count}")
print(f"总耗时: {total_time:.2f} 分钟")

--- 任务启动：共计 415 个样点 (多线程: True) ---
!!! OID 6 失败: ERROR 999999: 异常错误导致工具失败。 请联系 Esri 技术支持 (http://esriurl.com/support) 以报告漏洞，并参阅错误帮助以获取潜在解决方案或解决方法。
栅格数据集已存在
执行(ExtractByMask)失败。

!!! OID 4 失败: ERROR 999999: 异常错误导致工具失败。 请联系 Esri 技术支持 (http://esriurl.com/support) 以报告漏洞，并参阅错误帮助以获取潜在解决方案或解决方法。
栅格数据集已存在
执行(ExtractByMask)失败。

!!! OID 2 失败: ERROR 999999: 异常错误导致工具失败。 请联系 Esri 技术支持 (http://esriurl.com/support) 以报告漏洞，并参阅错误帮助以获取潜在解决方案或解决方法。
栅格数据集已存在
执行(ExtractByMask)失败。

!!! OID 1 失败: ERROR 999999: 异常错误导致工具失败。 请联系 Esri 技术支持 (http://esriurl.com/support) 以报告漏洞，并参阅错误帮助以获取潜在解决方案或解决方法。
栅格数据集已存在
执行(ExtractByMask)失败。

!!! OID 5 失败: ERROR 999999: 异常错误导致工具失败。 请联系 Esri 技术支持 (http://esriurl.com/support) 以报告漏洞，并参阅错误帮助以获取潜在解决方案或解决方法。
栅格数据集已存在
执行(ExtractByMask)失败。

!!! OID 9 失败: 执行失败。参数无效。
ERROR 000725: 输出影像服务器图层: 数据集 Raw_Elevation_Layer 已存在。
执行(MakeImageServerLayer)失败。

!!! OID 8 失败: 执行失败。参数无效。
ERROR 000725: 输出影像服务器图层: 数据集 Raw_Elevation_Layer 已存在。
执行(MakeImageServerLayer)失败。

!!! OID 7 失败: 执行失败。参数无效。
ERROR 00